In [1]:
%%capture
%cd /content
!rm -r mit6.S982
!git clone -b auction https://github.com/giannisdaras/mit6.S982.git
%cd /content/mit6.S982/projects/diffusion_auctions/Experiments/score_composition_reward/
%ls

In [45]:
import os, json, math, itertools, warnings, functools, argparse
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Union

import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors # For heatmaps
from google.colab import files

from transformers import CLIPModel, CLIPProcessor
from diffusers import FluxPipeline
from diffusers.image_processor import PipelineImageInput
from diffusers.pipelines.flux.pipeline_flux import retrieve_timesteps, calculate_shift
from diffusers.pipelines.flux.pipeline_output import FluxPipelineOutput
from diffusers.utils import is_torch_xla_available


DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
DTYPE           = torch.bfloat16 if torch.cuda.is_available() else torch.float16
CACHE_DIR   = "/n/netscratch/kempner_sham_lab/Lab/lilliansun/huggingface_cache"

DATA_PATH       = Path("../../evals/simple_eval/simple_eval.json")
# NUM_SAMPLES     = 5 # Removed global variable

GLOBAL_WORK = False
DO_TRUE_CFG = False

# CLIP Model Configuration
CLIP_MODEL_ID = "openai/clip-vit-large-patch14-336"

if is_torch_xla_available():
    import torch_xla.core.xla_model as xm
    XLA_AVAILABLE = True
else:
    XLA_AVAILABLE = False

class FluxPipelineAuction(FluxPipeline):
    @torch.no_grad()
    def __call__(
        self,
        agent1_prompt: Union[str, List[str]] = None,
        agent1_prompt_2: Optional[Union[str, List[str]]] = None,
        agent2_prompt: Union[str, List[str]] = None,
        agent2_prompt_2: Optional[Union[str, List[str]]] = None,
        joint_agents_prompt: Union[str, List[str]] = None,
        joint_agents_prompt_2: Optional[Union[str, List[str]]] = None,
        auction_w: float = 1.0,
        height: Optional[int] = None,
        width: Optional[int] = None,
        num_inference_steps: int = 28,
        sigmas: Optional[List[float]] = None,
        guidance_scale: float = 3.5,
        num_images_per_prompt: Optional[int] = 1,
        generator: Optional[Union[torch.Generator, List[torch.Generator]]] = None,
        latents: Optional[torch.FloatTensor] = None,
        agent1_prompt_embeds: Optional[torch.FloatTensor] = None,
        agent1_pooled_prompt_embeds: Optional[torch.FloatTensor] = None,
        agent2_prompt_embeds: Optional[torch.FloatTensor] = None,
        agent2_pooled_prompt_embeds: Optional[torch.FloatTensor] = None,
        joint_agents_prompt_embeds: Optional[torch.FloatTensor] = None,
        joint_agents_pooled_prompt_embeds: Optional[torch.FloatTensor] = None,
        ip_adapter_image: Optional[PipelineImageInput] = None,
        ip_adapter_image_embeds: Optional[List[torch.Tensor]] = None,
        negative_ip_adapter_image: Optional[PipelineImageInput] = None,
        negative_ip_adapter_image_embeds: Optional[List[torch.Tensor]] = None,
        output_type: Optional[str] = "pil",
        return_dict: bool = True,
        joint_attention_kwargs: Optional[Dict[str, Any]] = None,
        callback_on_step_end: Optional[Callable[[int, int, Dict], None]] = None,
        callback_on_step_end_tensor_inputs: List[str] = ["latents"],
        max_sequence_length: int = 512,
    ):
        height = height or self.default_sample_size * self.vae_scale_factor
        width = width or self.default_sample_size * self.vae_scale_factor

        self.check_inputs(
            agent1_prompt,
            agent1_prompt_2,
            height,
            width,
            negative_prompt=agent2_prompt,
            negative_prompt_2=agent2_prompt_2,
            prompt_embeds=agent1_prompt_embeds,
            negative_prompt_embeds=agent2_prompt_embeds,
            pooled_prompt_embeds=agent1_pooled_prompt_embeds,
            negative_pooled_prompt_embeds=agent2_pooled_prompt_embeds,
            callback_on_step_end_tensor_inputs=callback_on_step_end_tensor_inputs,
            max_sequence_length=max_sequence_length,
        )

        self._guidance_scale = guidance_scale
        self._joint_attention_kwargs = joint_attention_kwargs
        self._current_timestep = None
        self._interrupt = False

        if agent1_prompt is not None and isinstance(agent1_prompt, str):
            batch_size = 1
        elif agent1_prompt is not None and isinstance(agent1_prompt, list):
            batch_size = len(agent1_prompt)
        else:
            batch_size = agent1_prompt_embeds.shape[0]

        device = self._execution_device
        lora_scale = (
            self._joint_attention_kwargs.get("scale", None) if self._joint_attention_kwargs is not None else None
        )
        has_agent2_prompt = agent2_prompt is not None or (
            agent2_prompt_embeds is not None and agent2_pooled_prompt_embeds is not None
        )
        do_score_composition = has_agent2_prompt

        (
            agent1_prompt_embeds,
            agent1_pooled_prompt_embeds,
            text_ids,
        ) = self.encode_prompt(
            prompt=agent1_prompt,
            prompt_2=agent1_prompt_2,
            prompt_embeds=agent1_prompt_embeds,
            pooled_prompt_embeds=agent1_pooled_prompt_embeds,
            device=device,
            num_images_per_prompt=num_images_per_prompt,
            max_sequence_length=max_sequence_length,
            lora_scale=lora_scale,
        )

        if do_score_composition:
            (
                agent2_prompt_embeds,
                agent2_pooled_prompt_embeds,
                _,
            ) = self.encode_prompt(
                prompt=agent2_prompt,
                prompt_2=agent2_prompt_2,
                prompt_embeds=agent2_prompt_embeds,
                pooled_prompt_embeds=agent2_pooled_prompt_embeds,
                device=device,
                num_images_per_prompt=num_images_per_prompt,
                max_sequence_length=max_sequence_length,
                lora_scale=lora_scale,
            )
            (
                joint_agents_prompt_embeds,
                joint_agents_pooled_prompt_embeds,
                _,
            ) = self.encode_prompt(
                prompt=joint_agents_prompt,
                prompt_2=joint_agents_prompt_2,
                prompt_embeds=joint_agents_prompt_embeds,
                pooled_prompt_embeds=joint_agents_pooled_prompt_embeds,
                device=device,
                num_images_per_prompt=num_images_per_prompt,
                max_sequence_length=max_sequence_length,
                lora_scale=lora_scale,
            )

        num_channels_latents = self.transformer.config.in_channels // 4
        latents, latent_image_ids = self.prepare_latents(
            batch_size * num_images_per_prompt,
            num_channels_latents,
            height,
            width,
            agent1_prompt_embeds.dtype,
            device,
            generator,
            latents,
        )

        sigmas = np.linspace(1.0, 1 / num_inference_steps, num_inference_steps) if sigmas is None else sigmas
        image_seq_len = latents.shape[1]
        mu = calculate_shift(
            image_seq_len,
            self.scheduler.config.get("base_image_seq_len", 256),
            self.scheduler.config.get("max_image_seq_len", 4096),
            self.scheduler.config.get("base_shift", 0.5),
            self.scheduler.config.get("max_shift", 1.15),
        )
        timesteps, num_inference_steps = retrieve_timesteps(
            self.scheduler,
            num_inference_steps,
            device,
            sigmas=sigmas,
            mu=mu,
        )
        num_warmup_steps = max(len(timesteps) - num_inference_steps * self.scheduler.order, 0)
        self._num_timesteps = len(timesteps)

        if self.transformer.config.guidance_embeds:
            guidance_val = torch.full([1], self._guidance_scale, device=device, dtype=torch.float32)
            guidance = guidance_val.expand(latents.shape[0])
        else:
            guidance = None

        with self.progress_bar(total=num_inference_steps) as progress_bar:
            for i, t in enumerate(timesteps):
                if self.interrupt:
                    continue
                self._current_timestep = t
                timestep = t.expand(latents.shape[0]).to(latents.dtype)

                agent1_noise_pred = self.transformer(
                    hidden_states=latents,
                    timestep=timestep / 1000,
                    guidance=guidance,
                    pooled_projections=agent1_pooled_prompt_embeds,
                    encoder_hidden_states=agent1_prompt_embeds,
                    txt_ids=text_ids,
                    img_ids=latent_image_ids,
                    joint_attention_kwargs=self._joint_attention_kwargs,
                    return_dict=False,
                )[0]

                if do_score_composition:
                    agent2_noise_pred = self.transformer(
                        hidden_states=latents,
                        timestep=timestep / 1000,
                        guidance=guidance,
                        pooled_projections=agent2_pooled_prompt_embeds,
                        encoder_hidden_states=agent2_prompt_embeds,
                        txt_ids=text_ids,
                        img_ids=latent_image_ids,
                        joint_attention_kwargs=self._joint_attention_kwargs,
                        return_dict=False,
                    )[0]
                    joint_agents_noise_pred = self.transformer(
                        hidden_states=latents,
                        timestep=timestep / 1000,
                        guidance=guidance,
                        pooled_projections=joint_agents_pooled_prompt_embeds,
                        encoder_hidden_states=joint_agents_prompt_embeds,
                        txt_ids=text_ids,
                        img_ids=latent_image_ids,
                        joint_attention_kwargs=self._joint_attention_kwargs,
                        return_dict=False,
                    )[0]
                    if auction_w > 0.5:
                        weight_agent1 = 2 * auction_w - 1
                        noise_pred = (1 - weight_agent1) * joint_agents_noise_pred + weight_agent1 * agent1_noise_pred
                    elif auction_w < 0.5:
                        weight_agent2 = 1 - 2 * auction_w
                        noise_pred = (1 - weight_agent2) * joint_agents_noise_pred + weight_agent2 * agent2_noise_pred
                    else:
                        noise_pred = joint_agents_noise_pred
                else:
                    noise_pred = agent1_noise_pred

                latents_dtype = latents.dtype
                latents = self.scheduler.step(noise_pred, t, latents, return_dict=False)[0]

                if latents.dtype != latents_dtype:
                    if torch.backends.mps.is_available():
                        latents = latents.to(latents_dtype)

                if callback_on_step_end is not None:
                    callback_kwargs = {}
                    for k_cb in callback_on_step_end_tensor_inputs:
                        callback_kwargs[k_cb] = locals()[k_cb]
                    callback_outputs = callback_on_step_end(self, i, t, callback_kwargs)
                    latents = callback_outputs.pop("latents", latents)

                if i == len(timesteps) - 1 or ((i + 1) > num_warmup_steps and (i + 1) % self.scheduler.order == 0):
                    progress_bar.update()

                if XLA_AVAILABLE:
                    xm.mark_step()

        self._current_timestep = None

        if output_type == "latent":
            image = latents
        else:
            latents = self._unpack_latents(latents, height, width, self.vae_scale_factor)
            latents = (latents / self.vae.config.scaling_factor) + self.vae.config.shift_factor
            image = self.vae.decode(latents, return_dict=False)[0]
            image = self.image_processor.postprocess(image, output_type=output_type)

        self.maybe_free_model_hooks()
        if not return_dict:
            return (image,)
        return FluxPipelineOutput(images=image)

def generate_and_save_flux_images(pipe, data_item, idx, out_dir, bid1, bid2, num_samples):
    out_dir.mkdir(parents=True, exist_ok=True)
    agent1_raw = data_item.get("agent1_prompt","")
    agent2_raw = data_item.get("agent2_prompt","")
    base       = data_item.get("base_prompt","")
    joint   = f"{base} with {agent1_raw} and {agent2_raw}"
    a1p     = f"{base} with {agent1_raw}"
    a2p     = f"{base} with {agent2_raw}"
    auction_w = bid1 if (bid1 + bid2)>1e-6 else 0.5
    paths = []
    generated_new_images = False
    for s in range(num_samples):
        fn = out_dir / f"idx{idx:03d}_w{auction_w:.2f}_s{s:02d}.png"
        if not fn.exists():
            img = pipe(
                agent1_prompt=a1p,
                agent2_prompt=a2p,
                joint_agents_prompt=joint,
                auction_w=auction_w,
                guidance_scale=10.0,
                num_inference_steps=5
            ).images[0]
            img.save(fn)
            generated_new_images = True
        paths.append(fn)
    if generated_new_images:
        # print(f"Generated {len(paths)} images for prompt {idx} with auction weight {auction_w}")
        print(f"Example path: {fn}")
    return paths

def get_clip_alignment_scores(paths: List[str], text: str, model: CLIPModel, processor: CLIPProcessor) -> List[float]:
    """Calculates CLIP alignment scores for a list of image paths and a text prompt using batch processing."""
    all_scores = [0.0] * len(paths) # Initialize with default scores

    pil_images = []
    valid_indices = [] # Keep track of original indices of successfully loaded images

    for i, image_path in enumerate(paths):
        try:
            img = Image.open(image_path).convert("RGB")
            pil_images.append(img)
            valid_indices.append(i)
        except FileNotFoundError:
            print(f"Warning: Image file not found: {image_path}")
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")

    if not pil_images: # No images were successfully loaded
        return all_scores

    try:
        # Process text once
        text_inputs = processor(
            text=[text], # Process as a list even if it's a single text
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=processor.tokenizer.model_max_length
        ).to(DEVICE)
        text_inputs = {k: v.to(DTYPE if v.is_floating_point() and model.dtype == DTYPE else v.dtype) for k, v in text_inputs.items()}

        with torch.inference_mode():
            txt_feat = model.get_text_features(**text_inputs)
            txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)

            # Process all valid images in a batch
            image_inputs = processor(
                images=pil_images,
                return_tensors="pt"
            ).to(DEVICE)
            image_inputs = {k: v.to(DTYPE if v.is_floating_point() and model.dtype == DTYPE else v.dtype) for k, v in image_inputs.items()}

            img_feats_batch = model.get_image_features(**image_inputs)
            img_feats_batch = img_feats_batch / img_feats_batch.norm(dim=-1, keepdim=True)

            # Calculate scores for the batch
            batch_scores = (img_feats_batch @ txt_feat.T).squeeze()

            # Ensure batch_scores is iterable (it could be a 0-dim tensor if only one image)
            if batch_scores.ndim == 0:
                batch_scores_list = [batch_scores.item()]
            else:
                batch_scores_list = batch_scores.tolist()

        # Assign scores back to their original positions
        for i, score_val in enumerate(batch_scores_list):
            original_idx = valid_indices[i]
            all_scores[original_idx] = float(score_val)

    except Exception as e:
        print(f"Error during batched CLIP score calculation for prompt '{text}': {e}")
        # Scores for successfully loaded images might not be updated if error occurs here
        # Depending on desired robustness, one might want to revert to single processing or return all 0s.
        # For now, images that were part of the batch attempt but failed will retain 0.0

    return all_scores

def plot_current_utility_curves(records: List[Dict], output_dir: Path, p_idx_for_processing: int):
    output_dir.mkdir(parents=True, exist_ok=True)
    # print(f"Generating current utility plots in: {output_dir}")
    for (p_idx_plot, v1_plot, v2_plot), grp in itertools.groupby(
        sorted(records, key=lambda r:(r["prompt_index"], r["values"]["v1_true"],r["values"]["v2_true"])),
        key=lambda r:(r["prompt_index"], r["values"]["v1_true"],r["values"]["v2_true"])
    ):
        if p_idx_plot != p_idx_for_processing:
            continue

        group_records_sorted_by_b1_raw = sorted(list(grp), key=lambda r: r["bids"]["b1_raw"])

        xs_b1_raw = []
        ys_utility = []

        for r_item in group_records_sorted_by_b1_raw:
            b1_raw_val = r_item["bids"]["b1_raw"]

            xs_b1_raw.append(b1_raw_val)
            ys_utility.append(r_item["utility_1_best"])

        utilities_np = np.array(ys_utility)

        if not xs_b1_raw:
            continue

        plt.figure(figsize=(6,4))
        plt.plot(xs_b1_raw, utilities_np, marker="o", label="Utility₁ (Best Sample)")
        plt.title(f"Utility₁ vs Raw bid₁ (Prompt {p_idx_plot}, v₁={v1_plot:.2f}, v₂={v2_plot:.2f})")
        plt.xlabel("Raw bid₁"); plt.ylabel("Agent 1 Utility (Best Sample)")
        plt.grid(True); plt.legend()
        plot_filename = output_dir / f"prompt{p_idx_plot}_utility_v1_{v1_plot:.2f}_v2_{v2_plot:.2f}.png"
        plt.savefig(plot_filename, bbox_inches="tight"); plt.close()
    # print(f"Current utility plots for prompt {p_idx_for_processing} saved in {output_dir}")


def plot_optimal_bid_vs_true_value(records: List[Dict], output_dir: Path, p_idx: int,
                                   V1_TRUE_VALS: np.ndarray, B1_RAW_VALS: np.ndarray, v2_true_val: float):
    output_dir.mkdir(parents=True, exist_ok=True)
    # print(f"Generating Optimal Bid vs True Value plots in: {output_dir} (for v2_true={v2_true_val})")
    optimal_bids_for_v1 = []

    for v1_true_val in V1_TRUE_VALS:
        best_b1_raw, max_utility = None, -float('inf')
        relevant_records = [
            r for r in records
            if r["prompt_index"] == p_idx
            and math.isclose(r["values"]["v1_true"], v1_true_val)
            and math.isclose(r["values"]["v2_true"], v2_true_val)
        ]
        if not relevant_records: continue

        truthful_utility = -float('inf')
        for record_item in relevant_records:
            current_utility = record_item["utility_1_best"]
            current_b1_raw = record_item["bids"]["b1_raw"]
            if current_b1_raw == v1_true_val:
                truthful_utility = current_utility
            if not np.isnan(current_utility) and current_utility > max_utility:
                max_utility = current_utility
                best_b1_raw = current_b1_raw
        if math.isclose(max_utility, truthful_utility):
            best_b1_raw = v1_true_val

        if best_b1_raw is not None:
            optimal_bids_for_v1.append((v1_true_val, best_b1_raw))

    if not optimal_bids_for_v1:
        print(f"No data for plots for prompt {p_idx}, v2_true={v2_true_val}."); return

    v1_trues, b1_optimals = zip(*optimal_bids_for_v1)
    deviations = [opt - true for opt, true in zip(b1_optimals, v1_trues)]

    plt.figure(figsize=(7, 5)); plt.scatter(v1_trues, b1_optimals, label="Optimal b₁")
    min_val = min(min(v1_trues, default=0), min(b1_optimals, default=0)) - 0.1
    max_val = max(max(v1_trues, default=1), max(b1_optimals, default=1)) + 0.1
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', label="y=x (Perfect Truthfulness)")
    plt.xlabel("Agent 1 True Value (v₁)"); plt.ylabel("Agent 1 Optimal Raw Bid (b₁)")
    plt.title(f"Optimal b₁ vs v₁ (Prompt {p_idx}, v₂={v2_true_val:.2f})"); plt.legend(); plt.grid(True)
    plt.savefig(output_dir / f"prompt{p_idx}_optimal_b1_vs_v1_v2_{v2_true_val:.2f}.png", bbox_inches="tight"); plt.close()

    plt.figure(figsize=(7, 5)); plt.plot(v1_trues, deviations, marker='o', linestyle='-')
    plt.axhline(0, color='r', linestyle='--', label="Deviation = 0 (Perfect Truthfulness)")
    plt.xlabel("Agent 1 True Value (v₁)"); plt.ylabel("Deviation (Optimal b₁ - v₁)")
    plt.title(f"Optimal Bid Deviation (Prompt {p_idx}, v₂={v2_true_val:.2f})"); plt.legend(); plt.grid(True)
    plt.savefig(output_dir / f"prompt{p_idx}_optimal_b1_deviation_v2_{v2_true_val:.2f}.png", bbox_inches="tight"); plt.close()
    # print(f"Finished Optimal Bid vs True Value plots for prompt {p_idx}, v2_true={v2_true_val}.")

def plot_truthfulness_score_heatmap(records: List[Dict], output_dir: Path, p_idx: int,
                                   V1_TRUE_VALS: np.ndarray, B1_RAW_VALS: np.ndarray, V2_TRUE_VALS: np.ndarray):
    output_dir.mkdir(parents=True, exist_ok=True)
    # print(f"Generating Truthfulness REGRET Heatmap in: {output_dir}")
    deviation_score_map = np.full((len(V2_TRUE_VALS), len(V1_TRUE_VALS)), np.nan)

    for i, v2_true_val in enumerate(V2_TRUE_VALS):
        for j, v1_true_val in enumerate(V1_TRUE_VALS):
            best_b1_raw_for_cell = None # Initialize
            max_utility_for_cell = -float('inf') # Initialize and rename for clarity
            relevant_records = [
                r for r in records
                if r["prompt_index"] == p_idx
                and math.isclose(r["values"]["v1_true"], v1_true_val)
                and math.isclose(r["values"]["v2_true"], v2_true_val)
            ]
            if not relevant_records: continue

            # find truthful_utility and max_utility
            truthful_utility = None
            max_utility = -float("inf")

            for rec in relevant_records:
                util = rec["utility_1_best"]
                b1  = rec["bids"]["b1_raw"]

                if math.isclose(b1, v1_true_val, rel_tol=1e-6):
                    truthful_utility = util

                if not np.isnan(util) and util > max_utility:
                    max_utility = util

            # error if no truthful record
            if truthful_utility is None:
                raise ValueError(f"No truthful-utility record found for v1={v1_true_val}, v2={v2_true_val} (prompt {p_idx})")

            # error if truthful utility is zero (would divide by zero)
            if truthful_utility < 5e-3:
                # print("truthful_utility", truthful_utility)
                truthful_utility = 5e-3

                if max_utility < truthful_utility:
                    max_utility = truthful_utility

                # print("max_utility", max_utility)
                # if max_utility == 0.0:
                #     print("After setting truthful_utility to 1e-4, (max_utility - truthful_utility) / truthful_utility = 0")
                # else:
                #     print("After setting truthful_utility to 1e-4, (max_utility - truthful_utility) / truthful_utility = ", (max_utility - truthful_utility) / truthful_utility)
                # print("--------------------------------")
                # raise ZeroDivisionError(f"Truthful utility is zero for v1={v1_true_val}, v2={v2_true_val} (prompt {p_idx})")

            # after computing max_utility and truthful_utility…
            if max_utility == 0.0:
                deviation_score_map[i, j] = 0.0
            else:
                regret = (max_utility - truthful_utility) / truthful_utility
                if regret < 0:
                    # dump everything and crash
                    raise RuntimeError(
                        f"Negative regret detected at cell (i={i}, j={j}):\n"
                        f"  v1_true_val = {v1_true_val}\n"
                        f"  v2_true_val = {v2_true_val}\n"
                        f"  max_utility  = {max_utility}\n"
                        f"  truthful_util= {truthful_utility}\n"
                        f"  computed_regret = {regret}"
                    )
                deviation_score_map[i, j] = regret
    if np.all(np.isnan(deviation_score_map)):
        print(f"No data for Regret Score Heatmap for prompt {p_idx}.")
        return deviation_score_map # Return even if all NaN

    plt.figure(figsize=(8, 6))
    cmap = plt.colormaps["coolwarm_r"].copy(); cmap.set_bad(color='grey')
    plt.imshow(deviation_score_map, aspect='auto', origin='lower', cmap=cmap,
               extent=[min(V1_TRUE_VALS)-0.05, max(V1_TRUE_VALS)+0.05, min(V2_TRUE_VALS)-0.05, max(V2_TRUE_VALS)+0.05],
               vmin=0, vmax=1)
    plt.colorbar(label="Regret")
    plt.xlabel("Agent 1 True Value (v₁)")
    plt.ylabel("Agent 2 True Value (v₂)")
    plt.xticks(V1_TRUE_VALS)
    plt.yticks(V2_TRUE_VALS)
    plt.title(f"Regret Heatmap (Prompt {p_idx})")
    plt.savefig(output_dir / f"prompt{p_idx}_regret_heatmap.png", bbox_inches="tight"); plt.close()
    # print(f"Finished Regret Heatmap for prompt {p_idx}.")
    return deviation_score_map


def plot_regret_from_truthful_bidding_heatmap(records: List[Dict], output_dir: Path, p_idx: int,
                                             V1_TRUE_VALS: np.ndarray, B1_RAW_VALS: np.ndarray, v2_true_val: float):
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Generating Regret from Truthful Bidding Plot in: {output_dir} (for v2_true={v2_true_val})")

    regret_values_for_v1 = []

    for v1_true_iter in V1_TRUE_VALS:
        optimal_utility_for_this_v1 = -float('inf')
        truthful_utility_for_this_v1 = np.nan

        b1_truthful_val_for_v1 = B1_RAW_VALS[np.argmin(np.abs(B1_RAW_VALS - v1_true_iter))]

        possible_bids_for_this_v1 = [
            r for r in records
            if r["prompt_index"] == p_idx
            and math.isclose(r["values"]["v1_true"], v1_true_iter)
            and math.isclose(r["values"]["v2_true"], v2_true_val)
        ]

        if not possible_bids_for_this_v1:
            regret_values_for_v1.append((v1_true_iter, np.nan))
            continue

        found_truthful = False
        for rec_v1_bid_option in possible_bids_for_this_v1:
            current_util = rec_v1_bid_option["utility_1_best"]
            if not np.isnan(current_util):
                if current_util > optimal_utility_for_this_v1:
                    optimal_utility_for_this_v1 = current_util

                if math.isclose(rec_v1_bid_option["bids"]["b1_raw"], b1_truthful_val_for_v1):
                    truthful_utility_for_this_v1 = current_util
                    found_truthful = True

        regret = np.nan
        if optimal_utility_for_this_v1 > -float('inf') and found_truthful and not np.isnan(truthful_utility_for_this_v1):
             regret = max(0, optimal_utility_for_this_v1 - truthful_utility_for_this_v1)

        regret_values_for_v1.append((v1_true_iter, regret))

    if not regret_values_for_v1:
        print(f"No data for Regret Plot for prompt {p_idx}, v2_true={v2_true_val}."); return

    v1_trues, regrets = zip(*[(v, r) for v, r in regret_values_for_v1 if not np.isnan(r)])

    if not v1_trues:
        print(f"No valid regret data to plot for prompt {p_idx}, v2_true={v2_true_val}."); return

    plt.figure(figsize=(7, 5))
    plt.plot(v1_trues, regrets, marker='o', linestyle='-')
    plt.axhline(0, color='grey', linestyle='--', label="Regret = 0")
    plt.xlabel("Agent 1 True Value (v₁)")
    plt.ylabel("Regret (U_optimal - U_truthful)")
    plt.title(f"Regret from Truthful Bidding (Prompt {p_idx}, v₂={v2_true_val:.2f})")
    plt.legend(); plt.grid(True)
    plt.ylim(bottom=max(-0.01, min(regrets)-0.05 if regrets else -0.01))
    plot_filename = output_dir / f"prompt{p_idx}_regret_plot_v2_{v2_true_val:.2f}.png"
    plt.savefig(plot_filename, bbox_inches="tight"); plt.close()
    # print(f"Finished Regret Plot for prompt {p_idx}, v2_true={v2_true_val}.")


def calculate_utility(records: List[Dict]):
    if not records:
        return

    # a2_ref_samples are from images generated when agent 2 is dominant (e.g., b1=0, b2=1),
    # scored against agent 2's prompt. This list should be the same for all records
    # within a single prompt's analysis batch.
    a2_ref_samples_list = records[0]["alignments_scores"]["a2_ref_samples"]
    if not a2_ref_samples_list:
        warnings.warn("a2_ref_samples list is empty in the first record. Cannot robustly calculate a2_ref_best.")
        a2_ref_best = 0.0 # Default or could be np.nan if preferred for error propagation
    else:
        a2_ref_best = np.max(np.array(a2_ref_samples_list))

    for record in records:
        v1_true = record["values"]["v1_true"]
        v2_true = record["values"]["v2_true"]
        b1_raw  = record["bids"]["b1_raw"] # Agent 1's raw bid

        samp_scores_a1_np = np.array(record["alignments_scores"]["a1_samples"])
        samp_scores_a2_np = np.array(record["alignments_scores"]["a2_samples"])
        if samp_scores_a1_np.size == 0 or samp_scores_a2_np.size == 0:
            warnings.warn(
                f"Empty sample scores for record combination: "
                f"b1_raw={b1_raw}, v1_true={v1_true}, v2_true={v2_true}. "
                f"Setting utility to NaN."
            )
            record["utility_1_best"] = np.nan
            record["best_sample_idx"] = -1 # Indicates no valid sample found
            record["payment_1_calculated"] = np.nan
            continue
        # Determine the best sample based on maximizing a collective utility.
        # This collective utility uses Agent 1's raw bid (b1_raw) and Agent 2's true value (v2_true) as weights.
        collective_utilities = samp_scores_a1_np * b1_raw + samp_scores_a2_np * v2_true

        best_sample_idx = np.argmax(collective_utilities)
        record["best_sample_idx"] = int(best_sample_idx)

        # Scores of the chosen best sample
        a1_score_of_best_sample = samp_scores_a1_np[best_sample_idx]
        a2_score_of_best_sample = samp_scores_a2_np[best_sample_idx]
        # Calculate Agent 1's utility:
        # Utility1 = Value1(chosen_item) - Payment1
        # Value1(chosen_item) = a1_score_of_best_sample * v1_true
        # Payment1 = v2_true * ( externality_on_agent2 )
        # Externality_on_agent2 = a2_ref_best - a2_score_of_best_sample
        # This is how much "worse off" agent 2 is with the chosen sample compared to their best reference scenario.
        value_for_agent1 = a1_score_of_best_sample * v1_true
        payment_by_agent1 = v2_true * (a2_ref_best - a2_score_of_best_sample)

        utility_1_final = value_for_agent1 - payment_by_agent1

        record["utility_1_best"] = utility_1_final
        record["payment_1_calculated"] = payment_by_agent1 # Store for potential analysis

    return records
    # Records are modified in-place. No return value is strictly needed if the caller
    # uses the modified list, which is the current pattern.
    # The line `utility_dict = calculate_utility(...)` in main will assign None to utility_dict,
    # but utility_dict is not used thereafter.

Using device: cuda


## Actual Script to Run Below for each num sample and prompt index

In [3]:
print("Loading CLIP model and processor...")
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID, cache_dir=CACHE_DIR).to(dtype=DTYPE).to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID, cache_dir=CACHE_DIR)
print("CLIP model and processor loaded.")

Loading CLIP model and processor...


config.json:   0%|          | 0.00/4.76k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP model and processor loaded.


In [46]:
for num_samples_arg in [5, 15, 50]:
    for prompt_idx_arg in range(12):

        PLOT_TYPES_ROOT_DIR = Path(f"/content/mit6.S982/projects/diffusion_auctions/Experiments/score_composition_reward/utility_plots_trials_clip_mc_{num_samples_arg}")
        CURRENT_PLOTS_ROOT = PLOT_TYPES_ROOT_DIR / "current_utility_plots"
        TRUTH_PLOTS_APPROACH1_ROOT = PLOT_TYPES_ROOT_DIR / "truthfulness_optimal_vs_true"
        TRUTH_PLOTS_APPROACH3_ROOT = PLOT_TYPES_ROOT_DIR / "truthfulness_score_heatmap"
        TRUTH_PLOTS_APPROACH4_ROOT = PLOT_TYPES_ROOT_DIR / "truthfulness_regret_heatmap"

        # Path for the JSON file that will store heatmap averages
        TRUTHFULNESS_AVERAGES_JSON_PATH = PLOT_TYPES_ROOT_DIR / "truthfulness_averages.json"
        PLOT_TYPES_ROOT_DIR.mkdir(parents=True, exist_ok=True) # Ensure root directory exists

        NETSCRATCH_IMAGES_BASE_DIR = Path("/n/netscratch/kempner_sham_lab/Lab/lilliansun/diffusion_auctions/results/images_auctioned_trials")
        IMAGES_DIR_FOR_PROMPT = NETSCRATCH_IMAGES_BASE_DIR / f"prompt_{prompt_idx_arg}"
        IMAGES_DIR_FOR_PROMPT.mkdir(parents=True, exist_ok=True)
        print(f"Using image directory: {IMAGES_DIR_FOR_PROMPT}")

        json_output_dir_for_prompt = CURRENT_PLOTS_ROOT / f"prompt_{prompt_idx_arg}"
        json_output_dir_for_prompt.mkdir(parents=True, exist_ok=True)
        UTIL_JSON_PATH = json_output_dir_for_prompt / f"auction_utilities_prompt{prompt_idx_arg}.json"
        print(f"Utility JSON will be saved to: {UTIL_JSON_PATH}")

        with open(DATA_PATH) as f: items = json.load(f)
        if not (0 <= prompt_idx_arg < len(items)): print(f"Error: Prompt index {prompt_idx_arg} out of range.")
        current_item = items[prompt_idx_arg]
        p_idx_for_processing = prompt_idx_arg
        print(f"Processing prompt {p_idx_for_processing}: {current_item.get('base_prompt', '')}")

        # pipe = FluxPipelineAuction.from_pretrained("black-forest-labs/FLUX.1-schnell", torch_dtype=DTYPE, cache_dir=CACHE_DIR).to(DEVICE)

        B1_RAW_VALS = np.round(np.arange(0,1.01,0.1),2) # 0, 0.1, 0.2, ... , 1
        V2_TRUE_VALS = np.array([0.0, 0.25, 0.5, 0.75, 1.0])
        V1_TRUE_VALS = np.round(np.arange(0,1.01,0.1),2) # 0, 0.1, 0.2, ... , 1

        records = []
        # Check if the utility JSON file already exists
        if UTIL_JSON_PATH.exists():
            print(f"Attempting to load existing utility data from {UTIL_JSON_PATH}")
            try:
                with open(UTIL_JSON_PATH, 'r') as f:
                    loaded_records = json.load(f)

                # Basic validation: check if records exist and if they are for the current prompt
                if loaded_records and isinstance(loaded_records, list) and len(loaded_records) > 0:
                    if loaded_records[0].get("prompt_index") == p_idx_for_processing:
                        # Further validation: Check if num_samples matches (approx by checking one record's sample_paths)
                        # This is a heuristic. A more robust check might involve storing num_samples in the JSON.
                        if "sample_paths" in loaded_records[0] and len(loaded_records[0]["sample_paths"]) == num_samples_arg:
                            records = loaded_records
                            print(f"Successfully loaded {len(records)} records for prompt {p_idx_for_processing} with {num_samples_arg} samples from {UTIL_JSON_PATH}.")
                        else:
                            print(f"Warning: Number of samples in {UTIL_JSON_PATH} does not match requested {num_samples_arg}. Regenerating data.")
                    else:
                        print(f"Warning: Data in {UTIL_JSON_PATH} is for prompt {loaded_records[0].get('prompt_index')}, but requested {p_idx_for_processing}. Regenerating data.")
                else:
                    print(f"Warning: {UTIL_JSON_PATH} is empty or not in the expected list format. Regenerating data.")
            except json.JSONDecodeError:
                print(f"Error decoding JSON from {UTIL_JSON_PATH}. Regenerating data.")
            except Exception as e:
                print(f"An unexpected error occurred while loading {UTIL_JSON_PATH}: {e}. Regenerating data.")

        if not records: # If file didn't exist, was empty, invalid, for wrong prompt, or failed to load
            raise FileNotFoundError(f"Generating utility data for prompt {p_idx_for_processing} for {num_samples_arg} samples, as {UTIL_JSON_PATH} was not loaded or is invalid.")
            # This is the original data generation block
            ref_paths_agent2_dominant = generate_and_save_flux_images(pipe, current_item, p_idx_for_processing, IMAGES_DIR_FOR_PROMPT, 0.0, 1.0, num_samples_arg)
            ref_scores_a2_agent2_dominant_list = get_clip_alignment_scores(ref_paths_agent2_dominant, f"{current_item['base_prompt']} with {current_item['agent2_prompt']}", clip_model, clip_processor)

            records_for_prompt = []
            for b1_raw_iter, v2_true_iter in tqdm(itertools.product(B1_RAW_VALS, V2_TRUE_VALS), desc=f"Bids/Values for Prompt {p_idx_for_processing}", total=len(B1_RAW_VALS)*len(V2_TRUE_VALS)):
                if math.isclose(b1_raw_iter,0.0):
                    b1_norm, b2_norm = 0.0, 1.0
                    if math.isclose(v2_true_iter, 0.0):
                        b1_norm, b2_norm = 0.5, 0.5
                else:
                    tot = b1_raw_iter + v2_true_iter
                    if math.isclose(tot, 0.0):
                        b1_norm, b2_norm = 0.5, 0.5
                    else:
                        b1_norm, b2_norm = b1_raw_iter/tot, v2_true_iter/tot

                samp_paths = generate_and_save_flux_images(pipe, current_item, p_idx_for_processing, IMAGES_DIR_FOR_PROMPT, b1_norm, b2_norm, num_samples_arg)
                # Pass clip_model and clip_processor to get_clip_alignment_scores
                samp_scores_a1_list = get_clip_alignment_scores(samp_paths, f"{current_item['base_prompt']} with {current_item['agent1_prompt']}", clip_model, clip_processor)
                samp_scores_a2_list = get_clip_alignment_scores(samp_paths, f"{current_item['base_prompt']} with {current_item['agent2_prompt']}", clip_model, clip_processor)

                ref_scores_a2_np = np.array(ref_scores_a2_agent2_dominant_list)
                samp_scores_a1_np = np.array(samp_scores_a1_list)
                samp_scores_a2_np = np.array(samp_scores_a2_list)

                for v1_true_iter in V1_TRUE_VALS:
                    records_for_prompt.append({
                        "prompt_index": p_idx_for_processing,
                        "bids": dict(b1_raw=b1_raw_iter, b1_norm=b1_norm, b2_norm=b2_norm, v2_true_for_norm=v2_true_iter),
                        "values": dict(v1_true=v1_true_iter, v2_true=v2_true_iter),
                        "alignments_scores": dict(a1_samples=samp_scores_a1_np.tolist(), a2_samples=samp_scores_a2_np.tolist(), a2_ref_samples=ref_scores_a2_np.tolist()),
                        "sample_paths": [str(p) for p in samp_paths],
                    })
            records.extend(records_for_prompt)
            with open(UTIL_JSON_PATH,"w") as f: json.dump(records, f, indent=2)
            print(f"Saved all records for prompt {p_idx_for_processing} to {UTIL_JSON_PATH}")

        print("Cleaning up GPU memory after data generation/loading...") # Renamed message slightly
        # if 'pipe' in locals() and pipe is not None: del pipe; print("Deleted FLUX pipeline object.")

        # Explicitly delete CLIP model and processor loaded in main
        if 'clip_model' in locals() and clip_model is not None:
            del clip_model
            print("Deleted CLIP model instance.")
        if 'clip_processor' in locals() and clip_processor is not None:
            del clip_processor
            print("Deleted CLIP processor instance.")

        torch.cuda.empty_cache()
        print("CUDA cache emptied after data generation/loading.") # Renamed message slightly

        print(f"--- Generating all plots for prompt {p_idx_for_processing} (organized by plot type) ---")

        # Load existing averages data or initialize if not found
        averages_data = {}
        if TRUTHFULNESS_AVERAGES_JSON_PATH.exists():
            try:
                with open(TRUTHFULNESS_AVERAGES_JSON_PATH, 'r') as f:
                    averages_data = json.load(f)
            except json.JSONDecodeError:
                print(f"Warning: Could not decode JSON from {TRUTHFULNESS_AVERAGES_JSON_PATH}. Initializing fresh averages.")
                averages_data = {}

        # --- Start Modification: Filter records before plotting ---
        records_for_plotting = [r for r in records if r["prompt_index"] == p_idx_for_processing]
        if not records_for_plotting:
            print(f"Error: No records found for prompt {p_idx_for_processing} after generation/loading. Cannot generate plots.")
            # Depending on desired behavior, you might want to exit or return here.
            # For now, let it proceed, and the plot functions might print 'No data' messages.
        # --- End Modification ---

        records_for_plotting = calculate_utility(records_for_plotting) # calculate the utility based on best ref 2 image and best collective utility image

        output_dir_current = CURRENT_PLOTS_ROOT / f"prompt_{prompt_idx_arg}"
        plot_current_utility_curves(records_for_plotting, output_dir_current, p_idx_for_processing) # Pass filtered records

        output_dir_truth1 = TRUTH_PLOTS_APPROACH1_ROOT / f"prompt_{prompt_idx_arg}"
        for v2_true_val_plot in V2_TRUE_VALS:
            plot_optimal_bid_vs_true_value(records_for_plotting, output_dir_truth1, p_idx_for_processing, V1_TRUE_VALS, B1_RAW_VALS, v2_true_val_plot) # Pass filtered records

        output_dir_truth3 = TRUTH_PLOTS_APPROACH3_ROOT / f"prompt_{prompt_idx_arg}"
        # Get the deviation map and calculate its average
        deviation_score_map = plot_truthfulness_score_heatmap(records_for_plotting, output_dir_truth3, p_idx_for_processing, V1_TRUE_VALS, B1_RAW_VALS, V2_TRUE_VALS) # Pass filtered records
        if deviation_score_map is not None: # Check if map was returned
            avg_truthfulness_deviation = np.nanmean(deviation_score_map)
            if str(p_idx_for_processing) not in averages_data:
                averages_data[str(p_idx_for_processing)] = {}
            averages_data[str(p_idx_for_processing)]["truthfulness_deviation_avg"] = avg_truthfulness_deviation if not np.isnan(avg_truthfulness_deviation) else None # Store None if NaN
            print(f"Average Truthfulness Deviation for prompt {p_idx_for_processing}: {avg_truthfulness_deviation}")
        else:
            print(f"Could not calculate average truthfulness deviation for prompt {p_idx_for_processing} as heatmap data was not generated.")

        output_dir_truth4 = TRUTH_PLOTS_APPROACH4_ROOT / f"prompt_{prompt_idx_arg}"
        for v2_true_val_plot in V2_TRUE_VALS:
            plot_regret_from_truthful_bidding_heatmap(records_for_plotting, output_dir_truth4, p_idx_for_processing, V1_TRUE_VALS, B1_RAW_VALS, v2_true_val_plot) # Pass filtered records

        # Save the updated averages data
        try:
            with open(TRUTHFULNESS_AVERAGES_JSON_PATH, 'w') as f:
                json.dump(averages_data, f, indent=2)
            print(f"Truthfulness averages saved/updated at {TRUTHFULNESS_AVERAGES_JSON_PATH}")
        except Exception as e:
            print(f"Error saving truthfulness averages to {TRUTHFULNESS_AVERAGES_JSON_PATH}: {e}")

        print(f"--- All plots for prompt {p_idx_for_processing} generated. Base output directory: {PLOT_TYPES_ROOT_DIR} ---")

Using image directory: /n/netscratch/kempner_sham_lab/Lab/lilliansun/diffusion_auctions/results/images_auctioned_trials/prompt_0
Utility JSON will be saved to: /content/mit6.S982/projects/diffusion_auctions/Experiments/score_composition_reward/utility_plots_trials_clip_mc_5/current_utility_plots/prompt_0/auction_utilities_prompt0.json
Processing prompt 0: Two friends chatting over coffee at a cafe
Attempting to load existing utility data from /content/mit6.S982/projects/diffusion_auctions/Experiments/score_composition_reward/utility_plots_trials_clip_mc_5/current_utility_plots/prompt_0/auction_utilities_prompt0.json
Successfully loaded 605 records for prompt 0 with 5 samples from /content/mit6.S982/projects/diffusion_auctions/Experiments/score_composition_reward/utility_plots_trials_clip_mc_5/current_utility_plots/prompt_0/auction_utilities_prompt0.json.
Cleaning up GPU memory after data generation/loading...
CUDA cache emptied after data generation/loading.
--- Generating all plots fo

In [47]:
import shutil
from pathlib import Path
from google.colab import files

folders = [
    'utility_plots_trials_clip_mc_5',
    'utility_plots_trials_clip_mc_15',
    'utility_plots_trials_clip_mc_50'
]

base_dir = Path("/content/mit6.S982/projects/diffusion_auctions/Experiments/score_composition_reward")

for folder in folders:
    folder_path = base_dir / folder
    if not folder_path.exists():
        print(f"❌ Folder not found, skipping: {folder_path}")
        continue

    # Build absolute path for zip output
    zip_path = base_dir / f"{folder}.zip"

    # Create the zip archive
    #   make_archive takes: (base_name, format, root_dir, base_dir)
    #   base_name is without the “.zip”; it will append it automatically.
    archive_name = str(zip_path.with_suffix(''))
    shutil.make_archive(
        base_name=archive_name,
        format='zip',
        root_dir=str(folder_path.parent),
        base_dir=folder_path.name
    )
    print(f"✅ Created {zip_path}")

    # Download it
    files.download(str(zip_path))

✅ Created /content/mit6.S982/projects/diffusion_auctions/Experiments/score_composition_reward/utility_plots_trials_clip_mc_5.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Created /content/mit6.S982/projects/diffusion_auctions/Experiments/score_composition_reward/utility_plots_trials_clip_mc_15.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Created /content/mit6.S982/projects/diffusion_auctions/Experiments/score_composition_reward/utility_plots_trials_clip_mc_50.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [62]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

# --- set global font sizes once ---
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 16,
    'axes.labelsize': 16,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'legend.fontsize': 16,
    'legend.title_fontsize': 16
})

def plot_aggregated_regret_bar_graphs(base_path, output_dir):
    """Generates bar graphs for regret_deviation_avg."""
    base_path   = Path(base_path)
    output_dir  = Path(output_dir)

    # Define the order and labels for the Monte Carlo (mc) settings
    mc_settings_ordered = [
        {"label": "5", "dir_suffix": "5"},
        {"label": "15", "dir_suffix": "15"},
        # {"label": "25", "dir_suffix": "25"},
        {"label": "50", "dir_suffix": "50"},
    ]

    # Define the prompts to display and their new labels
    # Original prompt index : Display label
    prompt_display_mapping = {
        2: '1',
        3: '2',
        5: '3',
        6: '4',
        7: '5'
    }
    target_original_prompts = sorted(list(prompt_display_mapping.keys()))

    # Prepare a dict: { mc_label: { prompt_idx: regret_avg, … }, … }
    data_to_plot = {mc["label"]: {} for mc in mc_settings_ordered}
    all_prompt_indices = set()

    for mc_setting in mc_settings_ordered:
        regret_dir_name = f"utility_plots_trials_clip_mc_{mc_setting['dir_suffix']}"
        averages_file   = base_path / regret_dir_name / "truthfulness_averages.json"

        if averages_file.exists():
            try:
                with open(averages_file, 'r') as f:
                    content = json.load(f)
                for prompt_idx_str, values in content.items():
                    prompt_idx = int(prompt_idx_str)
                    all_prompt_indices.add(prompt_idx)
                    if "truthfulness_deviation_avg" in values:
                        data_to_plot[mc_setting["label"]][prompt_idx] = values["truthfulness_deviation_avg"]
            except json.JSONDecodeError:
                print(f"Warning: Could not decode JSON from {averages_file}")
            except Exception as e:
                print(f"Warning: Error processing {averages_file}: {e}")
        else:
            print(f"Warning: File not found {averages_file}")

    if not all_prompt_indices:
        print("No data found to plot for regret. Exiting.")
        return

    plot_prompts = target_original_prompts  # e.g. [2,3,5,6,7]
    display_labels = [prompt_display_mapping[p] for p in plot_prompts]

    num_prompts      = len(plot_prompts)
    num_mc_settings  = len(mc_settings_ordered)
    bar_width        = 0.8 / num_mc_settings

    plt.figure(figsize=(max(10, num_prompts * num_mc_settings * 0.5), 6))
    ax = plt.gca()
    for i, mc_setting in enumerate(mc_settings_ordered):
        mc_label = mc_setting["label"]
        # pull values only for your targets, in order
        values = [ data_to_plot[mc_label].get(p, np.nan) for p in plot_prompts ]
        x = (
            np.arange(num_prompts)
            + i * bar_width
            - (num_mc_settings * bar_width / 2)
            + bar_width / 2
        )
        ax.bar(x, values, width=bar_width, label=f"{mc_label}")

    ax.set_xlabel("Prompt Index")
    ax.set_ylabel("Average Regret")
    ax.set_title("Average Regret")
    ax.set_xticks(np.arange(num_prompts))
    ax.set_xticklabels(display_labels, rotation=0)
    ax.grid(True, axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()

    # move legend to the right side outside the plot
    ax.legend(title="Samples", loc='center left', bbox_to_anchor=(1, 0.5))

    ax.grid(True, axis='y', linestyle='--', alpha=0.7)

    # leave a little extra room on the right
    plt.tight_layout(rect=(0, 0, 1, 1))

    out_path = output_dir / "aggregated_regret_deviation_avg.png"
    plt.savefig(out_path)
    plt.close()
    print(f"Saved aggregated regret deviation plot to {out_path}")

    # zip and download folder
    files.download(out_path)

# Example call—now passing strings is fine, they’ll be converted to Path internally
plot_aggregated_regret_bar_graphs('/content/mit6.S982/projects/diffusion_auctions/Experiments/score_composition_reward', 'aggregated_regret_plots')

Saved aggregated regret deviation plot to aggregated_regret_plots/aggregated_regret_deviation_avg.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>